In [28]:
import pandas as pd

data_frame = pd.read_csv("/work/charite/notebooks/26/06/universities.csv").rename(
    {
        "Latitude": "coordinates.latitude",
        "Longitude": "coordinates.longitude",
        "Institution Name": "name",
        "City": "city",
        "Country": "country",
    },
    axis="columns",
)

In [29]:
import cartopy.crs
import geodatasets
import geopandas
import numpy as np
import shapely.geometry
from matplotlib import pyplot as plt

font_family = "Charité Text Office"
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [f"{font_family}"],
        "mathtext.fontset": "custom",
        "mathtext.rm": f"{font_family}",
        "mathtext.it": f"{font_family}:italic",
        "mathtext.bf": f"{font_family}:bold",
        "mathtext.bfit": f"{font_family}:bold:italic",
        "mathtext.sf": f"{font_family}",
    }
)

linewidth = 2.0

world = geopandas.read_file(geodatasets.get_path("naturalearth.land"))
clip_poly = geopandas.GeoSeries([shapely.geometry.box(-180, -60, 180, 90)], crs="EPSG:4326")
world = geopandas.clip(world, clip_poly)
projection = cartopy.crs.Robinson()
projected_world = world.to_crs(projection.proj4_init)

geo_data_frame = geopandas.GeoDataFrame(
    data_frame[["name"]],
    geometry=geopandas.points_from_xy(
        data_frame["coordinates.longitude"], data_frame["coordinates.latitude"]
    ),
    crs="EPSG:4326",
).to_crs(projection.proj4_init)

figure, axes = plt.subplots(nrows=1, ncols=1, figsize=(33.1, 7), dpi=1200, subplot_kw={"projection": projection})

axes.set_axis_off()

geo_axes = projected_world.plot(
    ax=axes,
    facecolor="whitesmoke",
    edgecolor="black",
    linewidth=linewidth / 2,
)

geo_data_frame.plot(
    ax=axes,
    kind="geo",
    categorical=True,
    legend=True,
    marker="o",
    markersize=50,
    edgecolor="navy",
    facecolor="orange",
    linewidth=linewidth / 2,
    zorder=3,
    label="HALFpipe users",
)

clip_poly = geopandas.GeoSeries([shapely.geometry.box(-170, -60, 180, 90)], crs="EPSG:4326")
west, _ = projection.transform_point(-140, 0, cartopy.crs.PlateCarree())
axes.set_xlim(left=west)

axes.legend()

figure.savefig("map.svg", bbox_inches="tight", pad_inches=0, transparent=True)

In [ ]:
import cartopy.crs
import geodatasets
import geopandas
import matplotlib.patches
import matplotlib.patheffects
import matplotlib.transforms
import numpy as np
import shapely.geometry
from matplotlib import pyplot as plt

linewidth = 2.0
connection_patch_kwargs = {
    "edgecolor": "none",
    "linewidth": linewidth,
    "arrowstyle": "wedge,tail_width=1.0,shrink_factor=0.5",
    # arrowstyle="fancy,head_length=0.9,head_width=0.9,tail_width=0.4",
    "alpha": 0.2,
    # shrinkB=8,
    "mutation_scale": 14,
}

world = geopandas.read_file(geodatasets.get_path("naturalearth.land"))
clip_poly = geopandas.GeoSeries(
    [shapely.geometry.box(-180, -60, 180, 90)], crs="EPSG:4326"
)
world = geopandas.clip(world, clip_poly)
projection = cartopy.crs.Robinson()
projected_world = world.to_crs(projection.proj4_init)

geo_data_frame = geopandas.GeoDataFrame(
    data_frame[["name"]],
    geometry=geopandas.points_from_xy(
        data_frame["coordinates.longitude"], data_frame["coordinates.latitude"]
    ),
    crs="EPSG:4326",
).to_crs(projection.proj4_init)

figure = plt.figure(figsize=(30, 15), dpi=1200)
grid_spec = figure.add_gridspec(ncols=3, nrows=1, wspace=0.05, width_ratios=[1, 2, 1])

map_axes = figure.add_subplot(grid_spec[0, 1], projection=projection)

map_axes.set_axis_off()
geo_axes = projected_world.plot(
    ax=map_axes,
    facecolor="whitesmoke",
    edgecolor="black",
    linewidth=linewidth / 2,
)

renderer = figure.canvas.get_renderer()  # pyright: ignore[reportAttributeAccessIssue]
min_x, min_y, max_x, max_y = projected_world.total_bounds
bounding_box = matplotlib.transforms.Bbox([[min_x, min_y], [max_x, max_y]])
bounding_box = bounding_box.transformed(map_axes.transData).transformed(
    figure.transFigure.inverted()
)
# bounding_box = map_axes.get_tightbbox(renderer).transformed(figure.transFigure.inverted())
left_axes = figure.add_axes(
    rect=(bounding_box.x0 - 0.14, bounding_box.y0, 0.15, bounding_box.height)
)
left_axes.set_xlim(0, 1)
left_axes.set_ylim(0, 1)
left_axes.set_axis_off()
left_axes.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

center = np.mean(map_axes.get_extent()[2:])  # pyright: ignore[reportAttributeAccessIssue]

for point in geo_data_frame.geometry:
    angle_b = 30 if point.y > center else -30  # pyright: ignore[reportAttributeAccessIssue]
    angle_a = 0
    # connection_patch = matplotlib.patches.ConnectionPatch(
    #     xyA=(0.5, 0.5),
    #     coordsA=left_axes.transAxes,
    #     xyB=(point.x, point.y),
    #     coordsB=map_axes.transData,
    #     color="navy",
    #     linewidth=0.0,
    #     connectionstyle=f"angle3,angleA={angle_a},angleB={angle_b}",
    #     arrowstyle="-|>",
    #     shrinkB=8,
    #     mutation_scale=20,
    # )
    # map_axes.add_artist(connection_patch)
    connection_patch = matplotlib.patches.ConnectionPatch(
        xyA=(0.5, 0.5),
        coordsA=left_axes.transAxes,
        xyB=(point.x, point.y),
        coordsB=map_axes.transData,
        connectionstyle=f"angle3,angleA={angle_a},angleB={angle_b}",
        facecolor="blueviolet",
        **connection_patch_kwargs,
    )
    map_axes.add_artist(connection_patch)

geo_data_frame.plot(
    ax=map_axes,
    kind="geo",
    categorical=True,
    legend=True,
    marker="o",
    markersize=50,
    edgecolor="navy",
    facecolor="orange",
    linewidth=linewidth / 2,
    zorder=3,
    label="HALFpipe users",
)
# figure.legend()

rectangle = matplotlib.patches.Rectangle(
    xy=(0.75, 0.25),
    width=0.5,
    height=0.5,
    edgecolor="black",
    facecolor="white",
    linewidth=linewidth / 2,
)

left_axes.add_artist(rectangle)
rectangle.set_clip_on(False)

left_axes.annotate(
    "Analysis plan",
    xy=(1.0, 0.5),
    fontsize="xx-large",
    horizontalalignment="center",
    verticalalignment="center",
)

# right_axes = figure.add_subplot(grid_spec[0, 2])
right_axes = figure.add_axes(
    rect=(bounding_box.x1 + 0.03, bounding_box.y0, 0.15, bounding_box.height)
)
right_axes.set_xlim(0, 1)
right_axes.set_ylim(0, 1)
right_axes.set_axis_off()

for point in geo_data_frame.geometry:
    angle_b = -30 if point.y > center else 30  # pyright: ignore[reportAttributeAccessIssue]
    angle_a = 0
    connection_patch = matplotlib.patches.ConnectionPatch(
        xyA=(0.5, 0.5),
        coordsA=right_axes.transAxes,
        xyB=(point.x, point.y),  # pyright: ignore[reportAttributeAccessIssue]
        coordsB=map_axes.transData,
        facecolor="darkturquoise",
        connectionstyle=f"angle3,angleA={angle_a},angleB={angle_b}",
        **connection_patch_kwargs,
    )
    # connection_patch.set_path_effects([
    #     matplotlib.patheffects.Stroke(linewidth=4.0, foreground="white"),
    #     matplotlib.patheffects.Normal(),
    # ])
    map_axes.add_artist(connection_patch)


rectangle = matplotlib.patches.Rectangle(
    xy=(-0.25, 0.25),
    width=0.5,
    height=0.5,
    edgecolor="black",
    facecolor="white",
    linewidth=linewidth / 2,
)
right_axes.add_artist(rectangle)
rectangle.set_clip_on(False)

right_axes.annotate(
    "Meta-analysis",
    xy=(0.0, 0.5),
    fontsize="xx-large",
    horizontalalignment="center",
    verticalalignment="center",
)

transform = figure.transFigure.inverted()
left_edge = transform.transform(left_axes.transAxes.transform((0.75, 0.5)))[0]
right_edge = transform.transform(right_axes.transAxes.transform((0.25, 0.5)))[0]

crop = matplotlib.transforms.Bbox([[left_edge, bounding_box.y0], [right_edge, bounding_box.y1]])
crop = crop.transformed(figure.transFigure).transformed(figure.dpi_scale_trans.inverted())
# Add outline
crop = crop.padded((linewidth / 2) / 72 / 2)

figure.savefig("collaborative.svg", bbox_inches=crop, transparent=True)

In [31]:
# figure, axes = plt.subplots(
#     nrows=1,
#     ncols=1,
#     figsize=(3, 3),
#     dpi=600,
# )

# axes.scatter(
#     [], [],
#     marker="o",
#     s=50,
#     edgecolor="navy",
#     facecolor="orange",
#     linewidth=linewidth / 2,
#     zorder=3,
#     label="HALFpipe users",
# )
# axes.set_axis_off()
# axes.legend()

# figure.savefig("legend.png", bbox_inches="tight")